# Who gets a seat in a lifeboat?

The Titanic passenger manifest gives us a real, imperfect prediction problem: can we predict whether a passenger survived using information that could have been recorded before the outcome? The point is not to recreate a historical decision or to make claims about people. The point is to practice a reliable modeling workflow when the data contain both numeric and categorical columns, missing values, and a tempting opportunity for data leakage.

By the end of this notebook, you should be able to:

1. identify a target, separate predictors from post-outcome information, and create a stratified train/test split;
2. choose preprocessing for numeric and categorical columns;
3. combine those preprocessing steps with a ColumnTransformer and a Pipeline;
4. evaluate a classifier against a simple baseline using a held-out test set and cross-validation; and
5. explain what your model can and cannot tell us about this dataset.

### What you will submit

Run the notebook from top to bottom after completing the tasks. Your submission should contain completed code cells, the required plot(s), and written responses in the marked response areas. Your responses should explain choices and observations, not just copy printed numbers.

## 0. Setup

The next cell contains access and display setup. The data-loading code is supplied so that your effort can focus on the sklearn workflow. fetch_openml downloads the public Titanic data the first time it runs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)


## 1. The data

The OpenML Titanic data describe passengers using fields such as class, age, sex, fare, and port of embarkation. The target is survived, coded as 0 or 1.

### Task 1 - Inspect before modeling

Run the next cell and look for:

- the shape of the data and the kinds of columns present;
- columns with missing values; and
- columns that might be unavailable until after the outcome.

Do not build a model yet. A careful inspection is part of the modeling workflow.


In [ ]:
titanic = fetch_openml(name="titanic", version=1, as_frame=True)
X_raw = titanic.data.copy()
y = pd.Series(titanic.target, name="survived").astype("int64")

print("Predictor shape:", X_raw.shape)
print("Target counts:")
print(y.value_counts().sort_index())
display(X_raw.head())


In [ ]:
display(X_raw.dtypes.to_frame("dtype"))
display(X_raw.isna().mean().sort_values(ascending=False).head(10).to_frame("missing_fraction"))


**Response 1.** Write two observations from your inspection. Include one observation about missingness or data types and one observation about a possible leakage or availability problem. Why would it be dangerous to let a model use information recorded after the outcome?

**Response:** _Write your answer here._


## 2. Define the prediction question and split the data

A useful first model should use only information plausibly available before a passenger boarded. Use this small, interpretable set as your starting point:

| Column | Kind | Why it is plausible before the outcome |
|---|---|---|
| pclass | numeric | ticket class |
| sex | categorical | passenger record |
| age | numeric | passenger record, with some missing values |
| sibsp, parch | numeric | family counts aboard |
| fare | numeric | ticket fare |
| embarked | categorical | embarkation port |

Fields such as boat and body are especially dangerous because they describe what happened after or during the outcome. We will leave them out. Limiting the first model to the table also keeps the preprocessing decisions visible.

### Task 2 - Build the modeling frame

Build the feature list from the table, then complete the train/test split using a 20% test set, RANDOM_STATE, and a stratification argument. Stratification keeps the class mix approximately comparable across the two splits.


In [ ]:
# TODO: list the pre-outcome columns from the table above.
feature_columns = []
X = X_raw[feature_columns].copy()

# TODO: create X_train, X_test, y_train, and y_test with train_test_split.
# Use the requested test size, random state, and a stratification argument.

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])


In [ ]:
split_check = pd.DataFrame({
    "all": y.value_counts(normalize=True),
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).sort_index()
display(split_check)


**Checkpoint.** The class proportions should be similar across the three columns above, although they may not be identical. If they are very different, revisit the stratify argument.

**Response 2.** In one or two sentences, explain why we split before fitting an imputer, scaler, or encoder. What information would leak if we fit those transformations on all rows first?

**Response:** _Write your answer here._


## 3. Establish a baseline

A model is only useful if it improves on a simple alternative. DummyClassifier(strategy="most_frequent") always predicts the training-set majority class. It is not meant to be clever; it gives us a reference point.

### Task 3 - Fit and evaluate the baseline

Use the estimator interface: fit on X_train, y_train, then predict only X_test. Compute both accuracy and balanced accuracy. Accuracy can hide weak performance on a less common class, while balanced accuracy gives each class equal weight.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")

# TODO: fit the baseline using only the training data
# TODO: create baseline_predictions for X_test

# TODO: calculate and print accuracy_score and balanced_accuracy_score


**Response 3.** Which metric seems more informative for this problem, and why? What would it mean if a more complicated model had accuracy only barely above the baseline?

**Response:** _Write your answer here._


## 4. Plan preprocessing by column type

Most sklearn estimators expect numeric input, but our table contains both numeric and categorical columns. We also do not want to discard rows with missing values just because a few fields are blank.

Use this plan as an API guide:

- Numeric columns: impute missing values with a statistic learned from the training data, then standardize the columns. SimpleImputer(strategy="median") and StandardScaler() are useful tools.
- Categorical columns: impute missing values with the most common category, then one-hot encode. Use OneHotEncoder(handle_unknown="ignore") so a category appearing later does not crash prediction.

### Task 4 - Classify the columns

Fill the two lists. Every column in X must occur exactly once: either in numeric_features or categorical_features. The assertions are a non-revealing check of that property.


In [ ]:
numeric_features = [
    # TODO: add the numeric column names from feature_columns
]
categorical_features = [
    # TODO: add the categorical column names from feature_columns
]

assert set(numeric_features).isdisjoint(categorical_features)
assert set(numeric_features + categorical_features) == set(X.columns)
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


### Task 5 - Make a numeric preprocessing pipeline

A pipeline is an ordered list of steps. Each non-final step must be a transformer with fit and transform. Here, the imputer must come before the scaler because the scaler cannot calculate means and standard deviations in the presence of missing values.

Use the numeric preprocessing plan above to replace the two placeholders.


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", None),  # TODO: choose the numeric imputer
    ("scaler", None),   # TODO: choose the numeric scaler
])


### Task 6 - Make a categorical preprocessing pipeline

Complete the categorical pipeline using the plan above. The encoder must be configured so that a category appearing in a test row or future row, but absent from a training fold, does not crash prediction.


In [ ]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", None),  # TODO: choose the categorical imputer
    ("onehot", None),   # TODO: choose and configure the encoder
])


### Task 7 - Combine branches with a ColumnTransformer

ColumnTransformer applies the numeric pipeline to one subset of columns and the categorical pipeline to another subset, then combines the results into one feature matrix. Complete the two transformer tuples below. Each tuple has the form (name, transformer, columns).


In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        # TODO: add a named tuple for numeric_pipeline and numeric_features
        # TODO: add a named tuple for categorical_pipeline and categorical_features
    ]
)


## 5. Assemble the first end-to-end pipeline

Now connect preprocessing to an estimator. The pipeline will learn imputation values, category levels, scaling parameters, and model coefficients during one call to fit. At prediction time, it applies the same learned transformations automatically.

### Task 8 - Fit logistic regression

Use logistic regression as the final estimator. Choose settings that make the solver converge and make your result reproducible; leave a brief code comment explaining those choices. Complete the pipeline, then fit and predict. Keep the test set untouched until this point.


In [ ]:
logistic_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("classifier", None),  # TODO: configure the requested estimator
])

# TODO: fit logistic_pipeline on the training data
# TODO: create logistic_predictions for X_test


In [ ]:
# TODO: calculate and print test accuracy and balanced accuracy for logistic_predictions

ConfusionMatrixDisplay.from_predictions(y_test, logistic_predictions)
plt.title("Logistic regression: held-out test set")
plt.show()


**Response 4.** Compare the logistic pipeline with your baseline. Use the confusion matrix to describe one type of error the model makes. Which class would you be especially concerned about misclassifying in a real application, and what additional context would you need before choosing a metric?

**Response:** _Write your answer here._


### Checkpoint: look inside the fitted transformer

One-hot encoding can create more columns than the original table had. That is expected: one categorical column can become several indicator columns. The following checks inspect the object you fitted without exposing a target answer.


In [ ]:
fitted_preprocess = logistic_pipeline.named_steps["preprocess"]
X_train_transformed = fitted_preprocess.transform(X_train)
X_test_transformed = fitted_preprocess.transform(X_test)
transformed_names = fitted_preprocess.get_feature_names_out()

print("Original columns:", X_train.shape[1])
print("Transformed columns:", X_train_transformed.shape[1])
print("First transformed names:", transformed_names[:10])
assert X_train_transformed.shape[0] == X_train.shape[0]
assert X_test_transformed.shape[0] == X_test.shape[0]
assert X_train_transformed.shape[1] == len(transformed_names)


**Response 5.** Why can the transformed feature count be larger than the original feature count? Explain in your own words how handle_unknown="ignore" changes prediction behavior for an unseen category.

**Response:** _Write your answer here._


## 6. Cross-validation without leakage

A single train/test split can make performance look unusually high or low. Cross-validation gives several training/validation splits. Because preprocessing is inside the pipeline, each fold fits its imputer, encoder, and scaler using only that fold's training rows.

### Task 9 - Estimate validation performance

Create a shuffled, stratified 5-fold splitter. Use cross_validate with logistic_pipeline and the full X, y data. Request at least accuracy and balanced_accuracy, and set return_train_score=True so you can compare training and validation performance. Do not call fit on X before cross-validation.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# TODO: supply the pipeline, data, splitter, metrics, and training-score option.
cv_results = cross_validate(
    # TODO
)

cv_summary = pd.DataFrame({
    "metric": ["accuracy", "balanced_accuracy"],
    "mean_validation": [cv_results["test_accuracy"].mean(), cv_results["test_balanced_accuracy"].mean()],
    "sd_validation": [cv_results["test_accuracy"].std(), cv_results["test_balanced_accuracy"].std()],
    "mean_training": [cv_results["train_accuracy"].mean(), cv_results["train_balanced_accuracy"].mean()],
})
display(cv_summary)


**Response 6.** Compare the mean validation scores with the held-out test scores. How much do results vary across folds? Is there evidence of a train/validation gap? Explain specifically how putting preprocessing inside logistic_pipeline protects this cross-validation estimate.

**Response:** _Write your answer here._


## 7. Reuse the preprocessing and swap the estimator

A useful pipeline separates a reusable data-preparation recipe from the final model. We will compare logistic regression with a random forest. The forest does not need standardized numeric values in the same way logistic regression does, but keeping the same preprocessing makes the comparison fair and keeps missing-value and categorical handling consistent.

### Task 10 - Build a second pipeline

Create forest_pipeline with the same preprocess object and a RandomForestClassifier. Use at least 200 trees and random_state=RANDOM_STATE; you may choose another forest parameter if you can explain it. The only pipeline step that should change is the final estimator.


In [ ]:
forest_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("classifier", None),  # TODO: RandomForestClassifier with the requested settings
])


### Task 11 - Compare models with the same folds

Use the existing cv splitter and cross_validate to compare validation accuracy and balanced accuracy for both pipelines. A short loop is appropriate here. Report a mean and a standard deviation for each metric and model. Do not select a winner from one fold alone.


In [ ]:
candidate_pipelines = {
    "logistic regression": logistic_pipeline,
    "random forest": forest_pipeline,
}

comparison_rows = []
for name, candidate in candidate_pipelines.items():
    # TODO: cross-validate candidate using cv and the two metrics; store the result in scores.
    comparison_rows.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "accuracy_sd": scores["test_accuracy"].std(),
        "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
        "balanced_accuracy_sd": scores["test_balanced_accuracy"].std(),
    })

comparison = pd.DataFrame(comparison_rows).set_index("model")
display(comparison.sort_values("balanced_accuracy_mean", ascending=False))


**Response 7.** Which pipeline would you choose for the final test-set check, and what is your reason? Consider the metric you care about, the size of the difference, and the fold-to-fold variability. If the models are close, describe a principled tie-breaker rather than treating a tiny difference as meaningful.

**Response:** _Write your answer here._


## 8. Final test-set check

The test set has been waiting while we made modeling decisions. Now choose one pipeline based on cross-validation, fit it once on X_train, y_train, and evaluate it on X_test. Do not use the test result to change the model and then report the same test result as if it were untouched.

### Task 12 - Evaluate your chosen pipeline

Assign either logistic_pipeline or forest_pipeline to chosen_pipeline. Then fit, predict, print a classification report, and display a confusion matrix. The classification report includes precision, recall, and F1 by class; connect those numbers to the error types visible in the matrix.


In [ ]:
# TODO: choose a pipeline using your cross-validation reasoning above
chosen_pipeline = None

# TODO: fit chosen_pipeline on the training data and create final_predictions for X_test

print(classification_report(y_test, final_predictions, target_names=["did not survive", "survived"]))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_predictions,
    display_labels=["did not survive", "survived"],
)
plt.title("Chosen pipeline: held-out test set")
plt.show()


## 9. Use the pipeline on new rows

A major practical benefit of a pipeline is that a new DataFrame can go through the same imputation, encoding, scaling, and prediction steps without manual preprocessing. The rows below are hypothetical and are included only to practice the prediction interface; they are not historical passengers.

### Task 13 - Make and interpret predictions

Run the cell and inspect both the predicted class and the estimated probability of survival. Then answer the response prompts. Remember that a probability from a model is not a guarantee or a causal explanation.


In [ ]:
new_passengers = pd.DataFrame({
    "pclass": [1, 3, 2],
    "sex": ["female", "male", "female"],
    "age": [29, np.nan, 35],
    "sibsp": [0, 1, 0],
    "parch": [0, 0, 1],
    "fare": [80.0, 8.0, 25.0],
    "embarked": ["S", "S", "Q"],
})

new_predictions = chosen_pipeline.predict(new_passengers)
new_probabilities = chosen_pipeline.predict_proba(new_passengers)[:, 1]

prediction_table = new_passengers.copy()
prediction_table["predicted_survival"] = new_predictions
prediction_table["estimated_survival_probability"] = new_probabilities
prediction_table


**Response 8.** Choose one hypothetical row. What information contributed to the model's prediction, and what important information is missing from this small feature set? Why would it be inappropriate to describe the prediction as a causal statement about the passenger?

**Response:** _Write your answer here._


## 10. Final synthesis

Complete the following in 5-8 sentences. Refer to your own outputs where useful.

1. Explain the role of the numeric pipeline, categorical pipeline, ColumnTransformer, and final estimator.
2. State how your chosen model compared with the baseline and with the other candidate.
3. Explain one way the workflow reduced leakage risk.


**Final response:** _Write your synthesis here._
